In [1]:
import os
import glob
import json
import re
import subprocess
from openai import OpenAI
import shutil
import hashlib

In [ ]:
from data.configuration import d4j_proj_base, code_base, d4j_command

In [3]:
import_json_path = "data/import_v2.json"
with open(import_json_path, "r") as file:
    import_json = json.load(file)
with open(os.path.join(code_base, "data/test_src.json"), "r") as f:
    content_path = json.load(f)

In [ ]:
%env OPENAI_API_KEY=<your_openai_api_key_here>

In [ ]:
# Retrieve the current PATH
current_path = os.environ.get('PATH')
print(f"Current PATH: {current_path}")

# Define the new directory you want to add
new_directory = '/path/to/defects4j/framework/bin'

# Update the PATH environment variable
os.environ['PATH'] = f"{new_directory}:{current_path}"

# Verify the update
updated_path = os.environ.get('PATH')
print(f"Updated PATH: {updated_path}")


In [6]:
# Set up OpenAI client
openai_client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

# Check if the API key is set
if not openai_client.api_key:
    raise ValueError("Please set the OPENAI_API_KEY environment variable.")


In [ ]:
# Quick test to confirm the API is working
print("Testing OpenAI API connection...")
try:
    # Make a simple API call
    test_completion = openai_client.chat.completions.create(
        model="o4-mini",  # Use "gpt-3.5-turbo" if GPT-4o is not available
        reasoning_effort="low",
        messages=[
            {"role": "user", "content": "Say 'This is a test'."}
        ],
        # temperature=0,
        # max_completion_tokens=5,
    )

    test_reply = test_completion.choices[0].message.content.strip()
    print(f"API Test Response: {test_reply}")
except Exception as e:
    print(f"OpenAI API test failed: {e}")
    raise e


Testing OpenAI API connection...


In [51]:
test_completion

ChatCompletion(id='chatcmpl-C2GcZeKLmsK1KYOtZ99Nm0PyMhHC4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='This is a test.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1754656583, model='o4-mini-2025-04-16', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=23, prompt_tokens=13, total_tokens=36, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [52]:
tgt_model = 'GPT-O4-MINI'
format = 'comment'
strategy = 'generation'
ablation = 'full'
version = 'buggy'
min_test_num = 10

In [53]:
# file_post_fix = f'unified_invoked'
file_post_fix = f'unified_invoked_docstring_prompt'
# file_post_fix = f'unified_invoked_advanced_docstring_prompt'
# file_post_fix = f'unified_invoked_docstring_code_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}_err_msg_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt'
# file_post_fix = f'unified_invoked_code_cot_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_generated_focal_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_generated_focal_prompt'
# file_post_fix = f'unified_invoked_generated_focal_err_msg_prompt_min_test_num_{min_test_num}'

In [54]:
# Define the path to the JSONL file
# jsonl_file_path = f'/path/to/repo/data/prompts/rq1/{tgt_model}_{format}_{strategy}_{ablation}_{version}_unified_invoked_generated_focal_err_msg_prompt_min_test_num_{min_test_num}.jsonl'
# jsonl_file_path = f'/path/to/repo/data/prompts/rq1/chatgpt_{format}_{strategy}_{ablation}_{version}_unified_invoked.jsonl'
jsonl_file_path = f'/path/to/repo/data/prompts/rq1/{tgt_model}_{format}_{strategy}_{ablation}_{version}_{file_post_fix}.jsonl'



In [ ]:
jsonl_file_path

In [56]:
deprecated_bugs = ["Cli_6_", "Closure_63_", "Closure_93_", "JacksonDatabind_65_", "JacksonDatabind_89_", "Lang_2_", "Lang_18_", "Lang_25_", "Lang_48_", "Time_21_"]

In [57]:
# Read the JSONL file and load the data
data = []
with open(jsonl_file_path, 'r') as f:
    for line in f:
        curr_json = json.loads(line)
        # check if curr_json['id'] starts with any of the deprecated_bugs
        if any(curr_json['id'].startswith(bug_id) for bug_id in deprecated_bugs):
            print(f"Skipping deprecated bug: {curr_json['id']}")
            continue
        data.append(curr_json)

# Check if data is loaded
if not data:
    raise ValueError("No data found in the JSONL file.")

print(f"Total items in data: {len(data)}")


Total items in data: 318


In [58]:
example_item = data[0]

In [59]:
# Parse project info
method_id = example_item['id']

In [60]:
print(example_item['focal_method'])

public String generateToolTipFragment(String toolTipText) {
        return " title=\"" + toolTipText
            + "\" alt=\"\"";
    }


In [61]:
# Use the content from the data as the prompt
prompt = example_item['content']

# Print the prompt being used
print("\nPrompt used for generating unit test:")
print(prompt)

# Generate unit test using OpenAI API
print(f"\nGenerating unit test for {method_id}")
try:
    chat_completion = openai_client.chat.completions.create(
        model="o4-mini",  # Use "gpt-3.5-turbo" if GPT-4 is not available
        reasoning_effort="high",
        messages=[
            {"role": "user", "content": prompt}
        ],
        # temperature=0.0,
        # max_tokens=8192
    )
except Exception as e:
    print(f"OpenAI API request failed: {e}")
    raise e

# Extract the assistant's reply
generated_test = chat_completion.choices[0].message.content.strip()
example_item['completion'] = generated_test


Prompt used for generating unit test:
You are a professional who writes Java test methods. Please help me write some unit tests in java language, details are listed below: 

```
// The following is the docstring of the method that you are going to test.

/**
 * Generates an HTML fragment for an image‐map or other tag that supplies
 * tooltip text via the title attribute while also providing an alt
 * attribute for accessibility compliance on <area> elements.
 *
 * <p>To produce valid and safe HTML, implementations must:
 * <ul>
 *   <li>Treat a null tooltip text as an empty string (i.e., no tooltip).</li>
 *   <li>Escape HTML special characters in the tooltip text
 *       (including &lt;, &gt;, &, ' and ").</li>
 *   <li>Include an alt attribute, which may be empty or mirror the
 *       tooltip text, to satisfy accessibility requirements.</li>
 * </ul>
 *
 * @param toolTipText  the raw text to be displayed in the tooltip;
 *                     may be {@code null} to indicate no too

In [62]:
print(example_item['completion'])

```java
package org.jfree.chart.imagemap;

import org.junit.Test;
import static org.junit.Assert.*;

public class StandardToolTipTagFragmentGeneratorTest {

    private final StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();

    @Test
    public void testConstructorCreatesInstance() {
        StandardToolTipTagFragmentGenerator g = new StandardToolTipTagFragmentGenerator();
        assertNotNull(g);
    }

    @Test
    public void testNullToolTip() {
        String frag = generator.generateToolTipFragment(null);
        assertNotNull(frag);
        assertEquals(" title=\"\" alt=\"\"", frag);
    }

    @Test
    public void testEmptyToolTip() {
        String frag = generator.generateToolTipFragment("");
        assertEquals(" title=\"\" alt=\"\"", frag);
    }

    @Test
    public void testSimpleToolTip() {
        String frag = generator.generateToolTipFragment("tooltip");
        assertEquals(" title=\"tooltip\" alt=\"tooltip\"", frag);
  

In [63]:
chat_completion

ChatCompletion(id='chatcmpl-C2GcbeCAljgwAZeilYUrVGNT4Q8FX', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```java\npackage org.jfree.chart.imagemap;\n\nimport org.junit.Test;\nimport static org.junit.Assert.*;\n\npublic class StandardToolTipTagFragmentGeneratorTest {\n\n    private final StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();\n\n    @Test\n    public void testConstructorCreatesInstance() {\n        StandardToolTipTagFragmentGenerator g = new StandardToolTipTagFragmentGenerator();\n        assertNotNull(g);\n    }\n\n    @Test\n    public void testNullToolTip() {\n        String frag = generator.generateToolTipFragment(null);\n        assertNotNull(frag);\n        assertEquals(" title=\\"\\" alt=\\"\\"", frag);\n    }\n\n    @Test\n    public void testEmptyToolTip() {\n        String frag = generator.generateToolTipFragment("");\n        assertEquals(" title=\\"\\" alt=\\"\\"", fra

In [64]:
chat_completion.usage.completion_tokens_details.reasoning_tokens

2496

In [65]:
code_blocks = re.findall(r'```java(.*?)```', example_item['completion'], re.DOTALL)

In [66]:
print(code_blocks[0])


package org.jfree.chart.imagemap;

import org.junit.Test;
import static org.junit.Assert.*;

public class StandardToolTipTagFragmentGeneratorTest {

    private final StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();

    @Test
    public void testConstructorCreatesInstance() {
        StandardToolTipTagFragmentGenerator g = new StandardToolTipTagFragmentGenerator();
        assertNotNull(g);
    }

    @Test
    public void testNullToolTip() {
        String frag = generator.generateToolTipFragment(null);
        assertNotNull(frag);
        assertEquals(" title=\"\" alt=\"\"", frag);
    }

    @Test
    public void testEmptyToolTip() {
        String frag = generator.generateToolTipFragment("");
        assertEquals(" title=\"\" alt=\"\"", frag);
    }

    @Test
    public void testSimpleToolTip() {
        String frag = generator.generateToolTipFragment("tooltip");
        assertEquals(" title=\"tooltip\" alt=\"tooltip\"", frag);
    }

  

In [67]:
example_json_path = f'/home/drixs2050/Documents/LLM4UT/data/prompts/rq1/{tgt_model}_{format}_{strategy}_{ablation}_{version}_{file_post_fix}_with_completion_example.jsonl'

In [68]:
# Write the example item json with completion to the file
with open(example_json_path, 'w') as f:
    f.write(json.dumps(example_item))
    f.write('\n')

In [69]:
# read the example json file to see if the completion is written
with open(example_json_path, 'r') as f:
    example_json = json.load(f)
    print(example_json['completion'])

```java
package org.jfree.chart.imagemap;

import org.junit.Test;
import static org.junit.Assert.*;

public class StandardToolTipTagFragmentGeneratorTest {

    private final StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();

    @Test
    public void testConstructorCreatesInstance() {
        StandardToolTipTagFragmentGenerator g = new StandardToolTipTagFragmentGenerator();
        assertNotNull(g);
    }

    @Test
    public void testNullToolTip() {
        String frag = generator.generateToolTipFragment(null);
        assertNotNull(frag);
        assertEquals(" title=\"\" alt=\"\"", frag);
    }

    @Test
    public void testEmptyToolTip() {
        String frag = generator.generateToolTipFragment("");
        assertEquals(" title=\"\" alt=\"\"", frag);
    }

    @Test
    public void testSimpleToolTip() {
        String frag = generator.generateToolTipFragment("tooltip");
        assertEquals(" title=\"tooltip\" alt=\"tooltip\"", frag);
  

In [70]:
len(data)

318

In [71]:
item_with_completion_path = f'/home/drixs2050/Documents/LLM4UT/data/prompts/rq1/{tgt_model}_{format}_{strategy}_{ablation}_{version}_{file_post_fix}_with_completion.jsonl'

In [ ]:
item_with_completion_path

In [73]:
failed_items = []

In [74]:
failed_items

[]

In [75]:
test_data = data

In [76]:
print(len([data_inst for data_inst in test_data if 'completion' in data_inst]))

1


In [77]:
# add tqdm for progress bar
from tqdm import tqdm

In [ ]:

with_completion = []
failed_items = []  # Initialize list for failed items
pbar = tqdm(test_data, desc="Processing items", unit="item", total=len(test_data))

# Loop through each item in the data
for item in pbar:
    # Parse project info
    if 'completion' in item:
        with_completion.append(item)
        continue
    method_id = item['id']
    prompt = item['content']
    if prompt:
        pbar.set_description(f"Generating unit test for {method_id}")

        chat_completion = None
        max_retries = 20  # Define maximum number of retries

        # Retry the API call up to max_retries times.
        for attempt in range(max_retries):
            try:
                # Generate unit test using OpenAI API
                chat_completion = openai_client.chat.completions.create(
                    model="o4-mini",
                    reasoning_effort="high",
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    # temperature=0.0,
                    # max_tokens=8192
                )
                # Print success message only if it wasn't the first attempt.
                if attempt > 0:
                    print(f"Retry success for method {method_id} after {attempt + 1} attempts")
                break  # Successful API call; exit the retry loop.
            except Exception as e:
                print(f"Attempt {attempt + 1} for method {method_id} failed: {e}")
                # Check if maximum retries have been reached
                if attempt == max_retries - 1:
                    print(f"Maximum retry attempts reached for method {method_id}. Breaking the loop.")
                    failed_items.append(item)
                    chat_completion = None

        # If no successful API call after all retries, break out of the outer loop.
        if chat_completion is None:
            break

        # Extract the assistant's reply
        generated_test = chat_completion.choices[0].message.content.strip()
        item['reasoning_token_count'] = chat_completion.usage.completion_tokens_details.reasoning_tokens
        item['completion'] = generated_test

    # assert 'completion' in item and not none and not empty
    assert 'completion' in item and item['completion'] is not None and item['completion'].strip() != ''

    # Append the item with completion to the list
    with_completion.append(item)

Generating unit test for Jsoup_37_buggy:  47%|████▋     | 149/318 [2:53:45<2:39:08, 56.50s/item]           

In [36]:
len(with_completion)

318

In [37]:
with_completion[0]['completion']

'```java\nimport org.jfree.chart.imagemap.StandardToolTipTagFragmentGenerator;\nimport org.junit.Test;\n\nimport static org.junit.Assert.*;\n\npublic class StandardToolTipTagFragmentGeneratorTest {\n\n    @Test\n    public void testConstructor() {\n        StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();\n        assertNotNull(generator);\n    }\n\n    @Test\n    public void testGenerateToolTipFragmentWithText() {\n        StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();\n        String result = generator.generateToolTipFragment("Hello");\n        assertEquals(" title=\\"Hello\\" alt=\\"\\"", result);\n    }\n\n    @Test\n    public void testGenerateToolTipFragmentWithEmptyString() {\n        StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();\n        String result = generator.generateToolTipFragment("");\n        assertEquals(" title=\\"\\" alt=\\"\\"", result);\

In [38]:
# Write the items with completion to the file
with open(item_with_completion_path, 'w') as f:
    for item in with_completion:
        f.write(json.dumps(item))
        f.write('\n')

In [ ]:
item_with_completion_path

In [40]:
failed_items

[]

In [41]:
# read the new json file to see if all the completions are written
with open(item_with_completion_path, 'r') as f:
    completion_json = []
    for line in f:
        completion_json.append(json.loads(line))
    print(len(completion_json))

for item in completion_json:
    if 'completion' not in item:
        print(item['id'])

318


In [42]:
print(completion_json[0]['completion'])

```java
import org.jfree.chart.imagemap.StandardToolTipTagFragmentGenerator;
import org.junit.Test;

import static org.junit.Assert.*;

public class StandardToolTipTagFragmentGeneratorTest {

    @Test
    public void testConstructor() {
        StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();
        assertNotNull(generator);
    }

    @Test
    public void testGenerateToolTipFragmentWithText() {
        StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();
        String result = generator.generateToolTipFragment("Hello");
        assertEquals(" title=\"Hello\" alt=\"\"", result);
    }

    @Test
    public void testGenerateToolTipFragmentWithEmptyString() {
        StandardToolTipTagFragmentGenerator generator = new StandardToolTipTagFragmentGenerator();
        String result = generator.generateToolTipFragment("");
        assertEquals(" title=\"\" alt=\"\"", result);
    }

    @Test
    public void t

In [43]:
# let LLM think about how to break the code